# HMS Forecast Summary

This report presents HMS forecast model results and performance metrics.

*Note: This is an experimental notebook viewer. The models referenced here are currently under development and are not operational.*

---

In [ ]:
# Parameters (papermill injectable)
S3_BUCKET = "flood-warning"
S3_PREFIX = "staging/temporary/results"
FORECAST_DATETIME = None  # override as "YYYY-MM-DD-HH", else uses current UTC hour


In [ ]:
import boto3
import os
import sys
import subprocess
from datetime import datetime, timezone
from pathlib import Path
from botocore.exceptions import ClientError
from IPython.display import Image, HTML, display
import pandas as pd

# Resolve forecast datetime
if FORECAST_DATETIME:
    dt = datetime.strptime(FORECAST_DATETIME, "%Y-%m-%d-%H").replace(tzinfo=timezone.utc)
else:
    dt = datetime.now(timezone.utc).replace(minute=0, second=0, microsecond=0)

init_time = dt.strftime("%Y-%m-%d-%H")
dated = f"{dt.year}/{dt.month:02d}/{dt.day:02d}/{dt.hour:02d}"
forcing_prefix      = f"staging/temporary/forcing/{dated}"
results_prefix      = f"{S3_PREFIX}/{dated}"
observations_prefix = f"staging/temporary/observations/{dated}"
print(f"Forecasts:     {init_time}")
print(f"Forcing:      s3://{S3_BUCKET}/{forcing_prefix}/")
print(f"Results:      s3://{S3_BUCKET}/{results_prefix}/")
print(f"Observations: s3://{S3_BUCKET}/{observations_prefix}/")

DATA_PATH   = "/notebooks/data"
MODEL_PATH  = "/notebooks/model"
OUTPUT_PATH = "/notebooks/work"

forcing_dir = Path(DATA_PATH) / "forcing"
results_dir = Path(MODEL_PATH) / "results"
obs_dir     = Path(DATA_PATH) / "observations"
for d in [forcing_dir, results_dir, obs_dir, Path(OUTPUT_PATH)]:
    d.mkdir(parents=True, exist_ok=True)

s3 = boto3.client("s3")
downloads = [
    (f"{forcing_prefix}/mrms_qpe.nc",           forcing_dir / "mrms_qpe.nc"),
    (f"{forcing_prefix}/hrrr_qpf.nc",           forcing_dir / "hrrr_qpf.nc"),
    (f"{results_prefix}/stats.parquet",         results_dir / "stats.parquet"),
    (f"{results_prefix}/forecast.parquet",      results_dir / "forecast.parquet"),
    (f"{results_prefix}/lookback.parquet",      results_dir / "lookback.parquet"),
    (f"{observations_prefix}/gages.parquet",    obs_dir / "gages.parquet"),
]
for key, local in downloads:
    print(f"  \u2193 {key.split('/')[-1]}", end=" ... ")
    try:
        s3.download_file(S3_BUCKET, key, str(local))
        print("ok")
    except ClientError as e:
        code = e.response['Error']['Code']
        print(f"SKIP ({code})")

Path(OUTPUT_PATH, "init_time.txt").write_text(init_time)

os.environ["DATA_PATH"]  = DATA_PATH
os.environ["MODEL_PATH"] = MODEL_PATH
os.environ["WORK_PATH"]  = OUTPUT_PATH
os.environ["OBS_PATH"]   = str(obs_dir)
print("\nReady.")

In [ ]:
# Run analysis script
print("Running HMS Analysis...\n")
result = subprocess.run([sys.executable, '/notebooks/hms_analysis.py'], cwd=OUTPUT_PATH)
print("\n" + "="*70)


## 1. Observed Precipitation

Total MRMS QPE accumulation over the lookback period — **left panel**: full warm up period; **right panel**: last 72 hours.


In [ ]:
img_file = f'{OUTPUT_PATH}/01_cumulative_forcing.png'
if Path(img_file).exists():
    display(Image(img_file))
else:
    print(f"Image not found: {img_file}")

## 2. Precipitation Forecast

High-Resolution Rapid Refresh (HRRR) Quantitative Precipitation Forecast animation showing predicted precipitation patterns over time.

In [ ]:
import base64
anim_file = f'{OUTPUT_PATH}/hrrr_forecast_animation.gif'
if Path(anim_file).exists():
    gif_b64 = base64.b64encode(open(anim_file, 'rb').read()).decode()
    display(HTML(f'<img src="data:image/gif;base64,{gif_b64}" style="max-width:100%;"/>'))
else:
    print(f'Animation not found')

## 3. Surface Water 

Time series comparison at selected stream gages and model junctions:
- **Black**: Observed flows
- **Blue**: Lookback modeled flows
- **Orange**: Forecast flows

In [ ]:
img_file = f'{OUTPUT_PATH}/02_hydrographs.png'
if Path(img_file).exists():
    display(Image(img_file))
else:
    print(f"Image not found")

## 4. HMS Performance Statistics

Observed vs. modeled flow metrics computed per basin element:

| Metric | Description |
|---|---|
| **Nash Sutcliffe** | 1 = perfect; 0 = mean baseline; <0 = worse than baseline |
| **Modified Kling-Gupta** | Composite of correlation, bias, and variability |
| **Correlation Coefficient** | Pearson r between observed and simulated |
| **Coefficient of Determination** | R² — fraction of variance explained |
| **Bias Ratio** | Ratio of simulated to observed volume |
| **Percent Bias** | % over-/under-estimation (positive = overestimate) |
| **RMSE Stdev** | RMSE normalised by observed std dev |

In [ ]:
stats_file = f'{OUTPUT_PATH}/stats_nash_sutcliffe.csv'
if Path(stats_file).exists():
    stats_df = pd.read_csv(stats_file)
    stats_df['Value'] = pd.to_numeric(stats_df['Value'], errors='coerce')
    print(f"Total Records: {len(stats_df)}")
    if 'StatisticType' in stats_df.columns and 'Value' in stats_df.columns:
        pivot = stats_df.pivot_table(index='BasinName', columns='StatisticType', values='Value')
        pivot.columns = [c.replace('Observed Flow ', '') for c in pivot.columns]
        pivot.index.name = 'Basin'
        pivot = pivot.round(4)
        display(pivot.style.background_gradient(cmap='RdYlGn', subset=['Nash Sutcliffe', 'Modified Kling-Gupta'], vmin=-1, vmax=1)
                      .background_gradient(cmap='RdYlGn_r', subset=['Percent Bias'], vmin=-50, vmax=50)
                      .format('{:.4f}', na_rep='—'))
    else:
        display(stats_df)
else:
    print('Stats file not found')